# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. Explore and sample the MedQuAD dataset
2. ***Generate ~3.5k medical multiple-choice questions (MCQs) using Phi-2***
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 2. Generate MCQs using Phi-2

In [3]:
import json
import re
from pathlib import Path
from tqdm import tqdm
from google.colab import files

# Upload JSONL file with question and answer fields
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving medquad_sampled.jsonl to medquad_sampled.jsonl


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "microsoft/phi-2"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Load and parse JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.float16).to(device)
model.eval()
print('Tokenizer and model loaded.')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Tokenizer and model loaded.


In [12]:
# Check if output file already exists
output_file = Path("generated_mcqs.jsonl")
if output_file.exists():
    response = input(f"File {output_file} exists. Delete it? (y/n): ")
    if response.lower().startswith("y"):
        output_file.unlink()
    else:
        raise RuntimeError("Aborting to avoid overwriting the file.")

# Format QA into prompt
def format_prompt(question, answer):
    return (
        f"""
          Convert the following medical Q&A into a multiple-choice question with one correct answer and three plausible but incorrect options.

          Provide your output in the following format:

          <question>
          [Multiple-choice question text with 4 labeled options: A., B., C., D.]
          </question>

          <response>
          [Short explanation of why the correct answer is right, optionally why others are wrong.]
          </response>

          <answer>
          [Letter of the correct answer: A, B, C, or D]
          </answer>

          Question: {question}
          Answer: {answer}
        """
    )

# Batch-generate MCQs
BATCH_SIZE = 8
results = []
for i in tqdm(range(0, BATCH_SIZE, BATCH_SIZE),
              desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):  # DEBUG
# for i in tqdm(range(0, len(qa_data), BATCH_SIZE,
#                     desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):
    batch = qa_data[i:i + BATCH_SIZE]
    prompts = [format_prompt(q["question"], q["answer"]) for q in batch]
    questions = [q["question"] for q in batch]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            # **inputs,
            input_ids=inputs["input_ids"],
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            attention_mask=inputs["attention_mask"]
        )

    decoded = tokenizer.batch_decode(outputs[:, len(inputs["input_ids"][0]) :],
                                     skip_special_tokens=True)
    # decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # for question, generated in zip(questions, decoded):
    #     results.append({"question": question, "answer": generated})
    for mcqa in decoded:
        q_match = re.search(r"<question>(.*?)</question>", mcqa)
        question = q_match.group(1) if q_match else None
        r_match = re.search(r"<response>(.*?)</response>", mcqa)
        response = r_match.group(1) if r_match else None
        a_match = re.search(r"<answer>(.*?)</answer>", mcqa)
        answer = r_match.group(1) if a_match else None
        if question and response and answer:
            results.append({"question": question, "response": response,
                            "answer": answer, "full_text": mcqa})

# Save to JSONL file
with open(output_file, "w") as f:
    for item in results:
        json.dump(item, f)
        f.write("\n")

print(f"Generated MCQs saved to {output_file}")

File generated_mcqs.jsonl exists. Delete it? (y/n): y


LLM Running on Micro Batches 8: 100%|██████████| 1/1 [00:17<00:00, 17.59s/it]

Generated MCQs saved to generated_mcqs.jsonl
